# Upwork API Technical Support Bot
### RAG Pipeline – Associate AI Developer Assignment


## 0. Setup – Load environment variables

In [ ]:
import os, time
from dotenv import load_dotenv

load_dotenv()  # reads .env file

DEEPINFRA_API_KEY  = os.getenv('DEEPINFRA_API_KEY')
DEEPINFRA_API_BASE = os.getenv('DEEPINFRA_API_BASE', 'https://api.deepinfra.com/v1/openai')
MODEL_NAME         = os.getenv('MODEL_NAME', 'meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo')
DOCS_PATH          = os.getenv('DOCS_PATH', './API_Documentation.pdf')

assert DEEPINFRA_API_KEY, 'Set DEEPINFRA_API_KEY in your .env file!'
print('Environment loaded ✅')

## Part A1 – Load document + Sanity Check

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(DOCS_PATH)
pages  = loader.load()

full_text   = ' '.join(p.page_content for p in pages)
total_chars = len(full_text)

print(f'Pages loaded   : {len(pages)}')
print(f'Total chars    : {total_chars:,}')
print(f'Sample text    :\n{full_text[:400]}')

## Part A2 – Chunking
> **Why overlap?** Code snippets (curl examples, JSON schemas) often span natural paragraph
> boundaries. A 50-character overlap ensures the tail of chunk N appears at the
> head of chunk N+1, so retrieval never returns a fragment that starts mid-example.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', ' ', ''],
)
chunks = splitter.split_documents(pages)
print(f'{len(pages)} pages → {len(chunks)} chunks')
print('\nFirst chunk preview:')
print(chunks[0].page_content)

## Part A3 – Embed & store in ChromaDB

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)

CHROMA_DIR = './chroma_db'
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)
print(f'Stored {len(chunks)} chunks in ChromaDB at "{CHROMA_DIR}" ✅')

## Part B1 – Semantic Retrieval

In [ ]:
def retrieve_top_chunks(query, k=3):
    retriever = vectorstore.as_retriever(search_kwargs={'k': k})
    return retriever.invoke(query)

test_query = 'How long is an OAuth access token valid for?'
docs = retrieve_top_chunks(test_query)
for i, d in enumerate(docs, 1):
    print(f'--- Chunk {i} (page {d.metadata.get("page","?")}) ---')
    print(d.page_content[:300])
    print()

## Part B2 – API Integration & Prompting

In [ ]:
from openai import OpenAI

SYSTEM_PROMPT = """You are a Senior Upwork API Consultant.
Answer ONLY using the documentation excerpts provided.
If the answer is not in the excerpts, say:
\"I'm sorry, but the provided documentation does not contain that information.\"
Never invent API details."""

client = OpenAI(api_key=DEEPINFRA_API_KEY, base_url=DEEPINFRA_API_BASE)

def answer_query(query):
    docs = retrieve_top_chunks(query)
    context = '\n\n'.join(f'[Excerpt {i+1}]\n{d.page_content}' for i, d in enumerate(docs))
    user_msg = f'Use ONLY these excerpts:\n{context}\n\nQuestion: {query}'
    
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role':'system','content':SYSTEM_PROMPT},
                  {'role':'user','content':user_msg}],
        max_tokens=512, temperature=0.1,
    )
    latency = time.time() - t0
    return resp.choices[0].message.content.strip(), latency, docs

print('Functions defined ✅')

## Part C – Evaluation (Ground Truth Questions)

In [ ]:
eval_questions = [
    'What is the specific request-per-second rate limit for the Upwork API, and is it enforced per Key or per IP?',
    'How long is an OAuth access token valid for?',
    'Can I use a Client Credentials Grant to access a user\'s private contract details?',
]

for q in eval_questions:
    print('='*65)
    print(f'Q: {q}')
    answer, latency, sources = answer_query(q)
    print(f'A: {answer}')
    print(f'Latency: {latency:.2f}s | Sources: {len(sources)} chunks')
    print()